# Titanic Dataset

## Setup

In [1]:
import pandas as pd
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import arff

In [2]:
filepath="data/Titanic.arff"
with open(filepath, "r", encoding="utf-8") as f:
    dataset = arff.load(f)
df = pd.DataFrame(dataset["data"],columns=[col[0] for col in dataset["attributes"]])

In [4]:
for col in df:
    print('Liczba etykiet zmiennej {}:{}'.format(col,len(df[col].unique())))

Liczba etykiet zmiennej pclass:3
Liczba etykiet zmiennej survived:2
Liczba etykiet zmiennej name:1307
Liczba etykiet zmiennej sex:2
Liczba etykiet zmiennej age:99
Liczba etykiet zmiennej sibsp:7
Liczba etykiet zmiennej parch:8
Liczba etykiet zmiennej ticket:929
Liczba etykiet zmiennej fare:282
Liczba etykiet zmiennej cabin:187
Liczba etykiet zmiennej embarked:4
Liczba etykiet zmiennej boat:28
Liczba etykiet zmiennej body:122
Liczba etykiet zmiennej home.dest:370


In [5]:
print('Liczba wsztystkich pasażerów:{}'.format(len(df)))

Liczba wsztystkich pasażerów:1309


Nasuwa się wniosek o zduplikwoanych pasażerach, wynik unique wskazuje na 1307 różnych imion, natomiast zbiór składa się z 1309 instancji.

In [11]:
df["name"].value_counts()

name
Connolly, Miss. Kate              2
Kelly, Mr. James                  2
Allen, Miss. Elisabeth Walton     1
Allison, Master. Hudson Trevor    1
Allison, Miss. Helen Loraine      1
                                 ..
Zabour, Miss. Hileni              1
Zabour, Miss. Thamine             1
Zakarian, Mr. Mapriededer         1
Zakarian, Mr. Ortin               1
Zimmerman, Mr. Leo                1
Name: count, Length: 1307, dtype: int64

In [9]:
df[df["name"].isin(["Connolly, Miss. Kate","Kelly, Mr. James"])]

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
725,3.0,1,"Connolly, Miss. Kate",female,22.0,0.0,0.0,370373,7.7500,NaN,Q,13,NaN,Ireland
726,3.0,0,"Connolly, Miss. Kate",female,30.0,0.0,0.0,330972,7.6292,NaN,Q,NaN,NaN,Ireland
924,3.0,0,"Kelly, Mr. James",male,34.5,0.0,0.0,330911,7.8292,NaN,Q,NaN,70.0,NaN
925,3.0,0,"Kelly, Mr. James",male,44.0,0.0,0.0,363592,8.0500,NaN,S,NaN,NaN,NaN


Patrząc po innych kolumnach wygląda jednak na to, że nie są to duplikaty tylko imiona Connolly, Miss. Kate Kelly oraz Mr. James się powtarzają.

## Komentarz do wyników (pkt 1–3)

Liczba wszystkich rekordów (pasażerów) w zbiorze danych wynosi **1309**. Porównując tę wartość z liczbą unikalnych etykiet dla poszczególnych zmiennych, możemy podzielić je na dwie główne grupy pod względem mocy zbioru (kardynalności):

### 1. Zmienne o małej kardynalności (niska moc zbioru)
Zmienne te przyjmują niewielką, ściśle ograniczoną liczbę unikalnych wartości.
* **`survived` (2 etykiety)** – zmienna binarna informująca o przeżyciu pasażera (0 lub 1).
* **`sex` (2 etykiety)** – podział na płeć (kobieta / mężczyzna).
* **`pclass` (3 etykiety)** – klasa biletu/podróży (1., 2. lub 3. klasa).
* **`embarked` (4 etykiety)** – port zaokrętowania (3 główne porty oraz brakujące wartości `NaN`).
* **`sibsp` (7 etykiet)** – liczba rodzeństwa lub małżonków na pokładzie.
* **`parch` (8 etykiet)** – liczba rodziców lub dzieci na pokładzie.
* **`boat` (28 etykiet)** – numer szalupy ratunkowej (zmienna o umiarkowanie małej kardynalności).

### 2. Zmienne o dużej kardynalności (wysoka moc zbioru)
Zmienne te posiadają bardzo dużo unikalnych wartości
* **`name` (1307 etykiet)** – niemal każdy pasażer posiada unikalne imię i nazwisko (różnica względem 1309 wynika z dwóch powtarzających się nazwisk).
* **`ticket` (929 etykiet)** – numer biletu; wiele osób ma indywidualne numery, choć część pasażerów podróżowała na jednym bilecie grupowym/rodzinnym.
* **`home.dest` (370 etykiet)** – miejsce zamieszkania / cel podróży pasażera (bardzo duże rozdrobnienie lokalizacji).
* **`fare` (282 etykiety)** – cena biletu (zmienna numeryczna ciągła, stąd duża liczba unikalnych kwot).
* **`cabin` (187 etykiet)** – numer kabiny pasażera (duża liczba unikalnych oznaczeń wynikająca z konkretnych numerów pokoi, mimo że pokładów jest znacznie mniej).
* **`body` (122 etykiety)** – numer identyfikacyjny odnalezionego ciała.

In [12]:
df["cabin"]

0            B5
1       C22 C26
2       C22 C26
3       C22 C26
4       C22 C26
         ...   
1304        NaN
1305        NaN
1306        NaN
1307        NaN
1308        NaN
Name: cabin, Length: 1309, dtype: str

In [13]:
df["CabinReduced"] = df["cabin"].astype(str).str[0]

In [16]:
df[["CabinReduced","cabin"]].head(20)

,CabinReduced,cabin
0,B,B5
1,C,C22 C26
2,C,C22 C26
3,C,C22 C26
4,C,C22 C26
5,E,E12
6,D,D7
7,A,A36
8,C,C101
9,NaN,NaN


In [17]:
for col in ["CabinReduced","cabin"]:
    print('Liczba etykiet zmiennej {}:{}'.format(col,len(df[col].unique())))

Liczba etykiet zmiennej CabinReduced:9
Liczba etykiet zmiennej cabin:187


In [22]:
print(f"Kardynalność kolumny cabin zmiejszono o {(187 - 9)/187 * 100:.2f}%.")

Kardynalność kolumny cabin zmiejszono o 95.19%.


In [24]:
print(df["CabinReduced"].isnull().mean())

0.774637127578304


## Uzasadnienie redukcji zmiennej `cabin`

Zmienna `cabin` jest jedną z niewielu cech tekstowych o wysokiej kardynalności (187 unikalnych wartości), która po odpowiednim przetworzeniu niesie za sobą bardzo istotną informację predykcyjną. Redukcja do pierwszej litery zmniejsza liczbę etykiet aż o **95,19%** (do 9 wartości) i pozwala wyodrębnić **oznaczenie pokładu** (np. A, B, C), na którym zakwaterowany był pasażer.

### Wpływ na przyszłe analizy i porównanie z innymi zmiennymi:
* **Wartość informacyjna:** Znajomość pokładu pozwala modelowi ocenić, jak daleko pasażer miał do szalup ratunkowych na górnym pokładzie oraz jak szybko jego kabina mogła zostać zalana. Ponadto pokład silnie koreluje ze statusem społecznym i klasą podróży.
* **Łatwiejsze modelowanie:** Zmniejszenie liczby kategorii ze 187 do 9 pozwala bezpiecznie zastosować kodowanie zmiennych kategorycznych (np. *One-Hot Encoding*) bez gwałtownego zwiększania liczby kolumn (unikamy tzw. przekleństwa wymiarowości).
* **Dlaczego nie inne zmienne o wysokiej kardynalności?** 
  * Zmienne takie jak `name` (imię i nazwisko) czy `ticket` (numer biletu) są unikalnymi identyfikatorami i w surowej formie wprowadzają jedynie szum.
  * Zmienna `body` (numer odnalezionego ciała) jest bezużyteczna w predykcji, ponieważ dotyczy wyłącznie ofiar katastrofy (powodowałaby tzw. wyciek danych – *data leakage*).
  * Zmienna `home.dest` (miejsce zamieszkania / cel podróży) jest zbyt rozdrobniona (370 etykiet), a informacje o zamożności czy trasie pasażera są już znacznie lepiej i czyściej ujęte w zmiennych `pclass` oraz `embarked`.
  * Zmienna `fare` (cena biletu) jest zmienną numeryczną ciągłą można by było co nawyżej wprowadzić podział na przedziały cenowe, np. tani, średni, drogi.

### Potencjalne negatywne skutki redukcji:
* **Utrata szczegółowej lokalizacji:** Pozostawiając samą literę pokładu, tracimy informację z numeru kabiny, która mogła wskazywać, czy pokój znajdował się na dziobie, śródokręciu czy rufie (co również miało znaczenie podczas zderzenia z górą lodową i tonięcia statku).
* **Traktowanie braków danych jako osobnej kategorii:** Mimo, że pozbywamy się szumu z rozdrbnionych to dalej duża część danych (77.5%) to dalej są braki które nic z soba nie niosą.